# LangChain 入门笔记本

欢迎！本笔记本带你用最少的代码跑通 LangChain 的核心流程：调用模型 → 提示词模板 → 串成链。

建议按顺序逐个单元格运行（Shift+Enter）。运行前请确保：
- 已激活 `.venv` 虚拟环境
- 已执行 `pip install -r requirements.txt`
- 已复制 `.env.example` 为 `.env` 并填入 `OPENAI_API_KEY`

> 详细概念讲解见 `docs/01-models-and-prompts.md` 与 `docs/02-chains.md`。


## 0. 环境检查

先确认能加载 `.env` 中的 API Key。


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.getenv('OPENAI_API_KEY'):
    raise SystemExit('✗ 未找到 OPENAI_API_KEY，请复制 .env.example 为 .env 并填写。')
print('✓ 环境就绪，API Key 已加载')


✓ 环境就绪，API Key 已加载


## 1. 你的第一个模型调用

`ChatOpenAI` 是对话模型统一的入口。直接调用 `.invoke('...')` 即可得到回复。


In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'))
response = llm.invoke('用一句话介绍 LangChain。')
print(response.content)


LangChain 是一个开源框架，旨在通过提供链式调用、数据连接、记忆管理和智能代理等模块化工具，简化基于大型语言模型的应用程序开发。


## 2. 用提示词模板管理输入

硬编码提示词不利于复用。`ChatPromptTemplate` 把可变部分参数化，支持 system / human 多角色消息。


In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ('system', '你是一个{role}，请用专业且简洁的语言回答。'),
    ('human', '{question}'),
])

print(prompt.format(role='Python 专家', question='什么是装饰器？'))


System: 你是一个Python 专家，请用专业且简洁的语言回答。
Human: 什么是装饰器？


## 3. 把「提示词 → 模型 → 解析」串成一条链（LCEL）

用管道符 `|` 把组件串起来，这便是 LangChain Expression Language（LCEL）。整条链本身也是一个可调用对象。


In [4]:
chain = prompt | llm | StrOutputParser()
print(chain.invoke({'role': 'Python 专家', 'question': '什么是装饰器？'}))


装饰器（Decorator）是 Python 中一种高阶函数，用于在不修改原有函数或类定义的前提下，动态地为其添加额外功能。其本质是接受一个可调用对象（函数/类）作为参数，并返回一个新的可调用对象（通常使用 `@` 语法糖）。典型应用包括日志记录、权限校验、性能计时、缓存等。例如：

```python
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print("Before call")
        result = func(*args, **kwargs)
        print("After call")
        return result
    return wrapper

@my_decorator
def say_hello(name):
    print(f"Hello, {name}")

say_hello("Alice")
# 输出：
# Before call
# Hello, Alice
# After call
```

装饰器也可以接受参数（需额外嵌套一层），或用于类（如 `@property`、`@classmethod`）。Python 内置了 `functools.wraps` 来保留原函数的元信息。


## 4. 动手改参数

把上面的 `role` 或 `question` 换成别的值再运行，体会模板的复用性。


In [ ]:
chain.invoke({'role': '历史老师', 'question': '简述第一次世界大战的起因。'})


## 5. 下一步

- 多轮对话与记忆：运行 `examples/03_memory.py`，参见 `docs/03-memory.md`
- 检索增强（RAG）：运行 `examples/04_rag.py`，参见 `docs/04-retrieval-and-rag.md`
- 智能代理：运行 `examples/05_agents.py`，参见 `docs/05-agents.md`

完成后，你可以在 `notebooks/` 中新建自己的笔记本继续练习。
